# Prefect 3.x — Introducción práctica

Este notebook es un recorrido por los conceptos fundamentales de Prefect usando ejemplos mínimos, igual que hicimos con MLflow. El objetivo es entender la herramienta antes de integrarla con el pipeline de churn.

Al terminar sabrás:
- Qué diferencia hay entre una función normal, un `@task` y un `@flow`
- Cómo funciona el logging en Prefect
- Qué son los retries y cómo se configuran
- Qué son los sub-flows y para qué sirven
- Cómo conectar al servidor y ver los runs en la UI

> **Sin servidor:** las primeras secciones funcionan sin Prefect Server — el flow corre como Python normal.  
> **Con servidor:** a partir de la sección 7 necesitas el stack levantado (`cd infra && docker compose -f docker-compose.yml -f docker-compose.prefect.yml up -d`). Prefect UI en http://localhost:4200

## 0. Setup

In [1]:
import random
import time

from prefect import flow, task
from prefect.logging import get_run_logger

import prefect
print(f"Prefect version: {prefect.__version__}")

Prefect version: 3.6.12


---
## 1. Conceptos fundamentales

Prefect tiene dos primitivas:

| Concepto | Qué es | Analogía |
|----------|--------|----------|
| `@task` | Una unidad de trabajo atómica. Se loggea, se reintenta, tiene estado propio. | Un paso de una receta |
| `@flow` | Un conjunto de tasks orquestadas. Es el punto de entrada, define el orden y las dependencias. | La receta completa |

**Reglas básicas:**
- Los flows llaman a tasks (y a otros flows)
- Las tasks NO llaman a otras tasks directamente — pasan el resultado como argumento
- Un flow puede correr sin servidor (como Python normal) o conectado a Prefect Server (aparece en la UI)

**Lo que Prefect añade sobre Python puro:**
- Cada task y flow tiene un **estado** (Running, Completed, Failed, Retrying)
- Los fallos se capturan, se loggean y se pueden reintentar automáticamente
- Conectado al servidor: historial de ejecuciones, gráfico de dependencias, alertas

---
## 2. Función normal vs @task vs @flow

El mismo código, tres formas de escribirlo.

In [2]:
# ── Versión 1: Python puro ──────────────────────────────────────────
def cuadrado(x):
    return x ** 2

def suma_cuadrados(valores):
    return sum(cuadrado(v) for v in valores)

resultado = suma_cuadrados([1, 2, 3, 4])
print(f"Python puro: {resultado}")

Python puro: 30


In [3]:
# ── Versión 2: con @task y @flow ────────────────────────────────────
@task(name="calcular-cuadrado")
def cuadrado_task(x: int) -> int:
    return x ** 2

@flow(name="suma-de-cuadrados", log_prints=True)
def suma_cuadrados_flow(valores: list[int]) -> int:
    cuadrados = [cuadrado_task(v) for v in valores]
    total = sum(cuadrados)
    print(f"Suma de cuadrados de {valores} = {total}")
    return total

suma_cuadrados_flow([1, 2, 3, 4])

20:23:15.723 | INFO    | Flow run 'masterful-armadillo' - Beginning flow run 'masterful-armadillo' for flow 'suma-de-cuadrados'

20:23:15.727 | INFO    | Flow run 'masterful-armadillo' - View at http://0.0.0.0:4200/runs/flow-run/6925c606-13d6-4312-b76b-7ece155cc3d0

20:23:15.738 | INFO    | Task run 'calcular-cuadrado-74e' - Finished in state Completed()

20:23:15.742 | INFO    | Task run 'calcular-cuadrado-7f5' - Finished in state Completed()

20:23:15.746 | INFO    | Task run 'calcular-cuadrado-fd7' - Finished in state Completed()

20:23:15.749 | INFO    | Task run 'calcular-cuadrado-a28' - Finished in state Completed()

20:23:15.751 | INFO    | Flow run 'masterful-armadillo' - Suma de cuadrados de [1, 2, 3, 4] = 30

20:23:15.767 | INFO    | Flow run 'masterful-armadillo' - Finished in state Completed()

30

El resultado es el mismo. La diferencia es lo que pasa por debajo:
- Cada llamada a `cuadrado_task()` crea un **Task Run** con su propio estado
- `suma_cuadrados_flow()` crea un **Flow Run** que los contiene
- Si alguna task falla, Prefect lo captura sin crashear todo el proceso
- Conectado al servidor, todo esto aparece en la UI con tiempos, logs y estados

---
## 3. Logging — get_run_logger()

Dentro de tasks y flows, el logger de Prefect es preferible a `print()` porque:
- Incluye el nombre del task/flow, el timestamp y el nivel de log
- Los logs aparecen en la UI de Prefect por run
- `log_prints=True` en el flow redirige los `print()` al logger de Prefect automáticamente

In [4]:
@task(name="validar-dato")
def validar(valor: float, minimo: float = 0.0) -> float:
    logger = get_run_logger()
    if valor < minimo:
        logger.warning(f"Valor {valor} está por debajo del mínimo {minimo}")
    else:
        logger.info(f"Valor {valor} validado correctamente")
    return max(valor, minimo)


@flow(name="demo-logging", log_prints=True)
def flow_logging():
    print("Inicio del flow")   # log_prints=True → aparece en los logs de Prefect
    v1 = validar(42.0)
    v2 = validar(-5.0, minimo=0.0)
    print(f"Valores procesados: {v1}, {v2}")


flow_logging()

20:23:17.506 | INFO    | Flow run 'orthodox-coucal' - Beginning flow run 'orthodox-coucal' for flow 'demo-logging'

20:23:17.507 | INFO    | Flow run 'orthodox-coucal' - View at http://0.0.0.0:4200/runs/flow-run/7075578d-68f6-4080-9338-a26f2c710d64

20:23:17.508 | INFO    | Flow run 'orthodox-coucal' - Inicio del flow

20:23:17.513 | INFO    | Task run 'validar-dato-a66' - Valor 42.0 validado correctamente

20:23:17.515 | INFO    | Task run 'validar-dato-a66' - Finished in state Completed()

20:23:17.519 | WARNING | Task run 'validar-dato-061' - Valor -5.0 está por debajo del mínimo 0.0

20:23:17.521 | INFO    | Task run 'validar-dato-061' - Finished in state Completed()

20:23:17.523 | INFO    | Flow run 'orthodox-coucal' - Valores procesados: 42.0, 0.0

20:23:17.581 | INFO    | Flow run 'orthodox-coucal' - Finished in state Completed()

---
## 4. Parámetros del flow

Los flows aceptan parámetros como cualquier función Python. Cuando se despliegan como deployments, esos parámetros se pueden pasar desde la UI o la CLI sin cambiar el código.

In [5]:
@task(name="procesar-elemento")
def procesar(elemento: str, mayusculas: bool) -> str:
    return elemento.upper() if mayusculas else elemento.lower()


@flow(name="flow-parametrizado", log_prints=True)
def flow_con_params(
    elementos: list[str] = ["hola", "mundo"],
    mayusculas: bool = False,
):
    resultados = [procesar(e, mayusculas) for e in elementos]
    print(f"Resultado: {resultados}")
    return resultados


# Llamada con parámetros por defecto
flow_con_params()

# Llamada con parámetros distintos — mismo código, comportamiento diferente
flow_con_params(elementos=["prefect", "mlflow", "docker"], mayusculas=True)

20:23:17.639 | INFO    | Flow run 'woodoo-wrasse' - Beginning flow run 'woodoo-wrasse' for flow 'flow-parametrizado'

20:23:17.640 | INFO    | Flow run 'woodoo-wrasse' - View at http://0.0.0.0:4200/runs/flow-run/c9e335d0-186a-4f8c-860f-10707341858f

20:23:17.643 | INFO    | Task run 'procesar-elemento-0fa' - Finished in state Completed()

20:23:17.647 | INFO    | Task run 'procesar-elemento-c2b' - Finished in state Completed()

20:23:17.648 | INFO    | Flow run 'woodoo-wrasse' - Resultado: ['hola', 'mundo']

...

20:23:17.690 | INFO    | Task run 'procesar-elemento-c10' - Finished in state Completed()

20:23:17.693 | INFO    | Task run 'procesar-elemento-f2b' - Finished in state Completed()

20:23:17.694 | INFO    | Flow run 'satisfied-petrel' - Resultado: ['PREFECT', 'MLFLOW', 'DOCKER']

20:23:17.708 | INFO    | Flow run 'satisfied-petrel' - Finished in state Completed()

['PREFECT', 'MLFLOW', 'DOCKER']

---
## 5. Retries — tolerancia a fallos

Una de las principales razones para usar Prefect en producción: las tasks que fallan se pueden reintentar automáticamente sin código extra. Útil para llamadas a APIs externas, descargas de datos, o cualquier operación que pueda fallar por razones transitorias.

In [6]:
intentos = {"count": 0}

@task(name="llamada-api-inestable", retries=3, retry_delay_seconds=1)
def llamada_api() -> str:
    logger = get_run_logger()
    intentos["count"] += 1
    logger.info(f"Intento #{intentos['count']}")

    # Simular fallo en los 2 primeros intentos
    if intentos["count"] < 3:
        logger.warning("Fallo transitorio — reintentando...")
        raise ConnectionError("Servicio no disponible (simulado)")

    logger.info("Conexión establecida")
    return "datos_recibidos"


@flow(name="demo-retries", log_prints=True)
def flow_retries():
    resultado = llamada_api()
    print(f"Resultado tras {intentos['count']} intentos: {resultado}")


flow_retries()

20:23:17.753 | INFO    | Flow run 'tidy-platypus' - Beginning flow run 'tidy-platypus' for flow 'demo-retries'

20:23:17.753 | INFO    | Flow run 'tidy-platypus' - View at http://0.0.0.0:4200/runs/flow-run/ce317709-f974-4d5f-89af-31c4b60dd20f

20:23:17.758 | INFO    | Task run 'llamada-api-inestable-7a3' - Intento #1

20:23:17.759 | WARNING | Task run 'llamada-api-inestable-7a3' - Fallo transitorio — reintentando...

20:23:17.760 | INFO    | Task run 'llamada-api-inestable-7a3' - Task run failed with exception: ConnectionError('Servicio no disponible (simulado)') - Retry 1/3 will start 1 second(s) from now

...

20:23:19.774 | INFO    | Task run 'llamada-api-inestable-7a3' - Intento #3

20:23:19.775 | INFO    | Task run 'llamada-api-inestable-7a3' - Conexión establecida

20:23:19.777 | INFO    | Task run 'llamada-api-inestable-7a3' - Finished in state Completed()

20:23:19.779 | INFO    | Flow run 'tidy-platypus' - Resultado tras 3 intentos: datos_recibidos

20:23:19.799 | INFO    | Flow run 'tidy-platypus' - Finished in state Completed()

Con `retries=3, retry_delay_seconds=1`:
- Si la task falla, Prefect espera 1 segundo y lo intenta de nuevo
- Hasta 3 reintentos adicionales (4 intentos totales)
- Si los 3 reintentos fallan, el task run pasa a estado `Failed` y el flow también
- En la UI de Prefect se ve cada intento por separado con sus logs

---
## 6. Sub-flows — flows dentro de flows

Un flow puede llamar a otro flow como si fuera una función normal. El flow hijo aparece anidado en la UI, con su propio task graph y sus propios logs. Útil para componer pipelines complejos a partir de bloques más simples y reutilizables.

In [7]:
@task(name="transformar")
def transformar(valor: float, factor: float) -> float:
    logger = get_run_logger()
    resultado = valor * factor
    logger.info(f"{valor} × {factor} = {resultado}")
    return resultado


@flow(name="procesar-lote", log_prints=True)
def procesar_lote(datos: list[float], factor: float) -> list[float]:
    """Sub-flow: procesa un lote de datos."""
    print(f"Procesando lote de {len(datos)} elementos con factor {factor}")
    return [transformar(d, factor) for d in datos]


@flow(name="pipeline-principal", log_prints=True)
def pipeline_principal():
    """Flow principal que coordina dos sub-flows."""
    print("Iniciando pipeline")

    # Llamada a sub-flows — aparecen anidados en la UI
    lote_a = procesar_lote([1.0, 2.0, 3.0], factor=2.0)
    lote_b = procesar_lote([4.0, 5.0, 6.0], factor=0.5)

    total = sum(lote_a) + sum(lote_b)
    print(f"Total combinado: {total}")
    return total


pipeline_principal()

20:23:19.845 | INFO    | Flow run 'portable-malkoha' - Beginning flow run 'portable-malkoha' for flow 'pipeline-principal'

20:23:19.849 | INFO    | Flow run 'portable-malkoha' - View at http://0.0.0.0:4200/runs/flow-run/4328ccd0-5102-4054-b08b-7f242a49718c

20:23:19.850 | INFO    | Flow run 'portable-malkoha' - Iniciando pipeline

20:23:19.935 | INFO    | Flow run 'acoustic-raccoon' - Beginning subflow run 'acoustic-raccoon' for flow 'procesar-lote'

20:23:19.936 | INFO    | Flow run 'acoustic-raccoon' - View at http://0.0.0.0:4200/runs/flow-run/a70a6e33-9460-4cba-a8d4-9a29ddaf0f94

...

20:23:20.031 | INFO    | Task run 'transformar-9d7' - Finished in state Completed()

20:23:20.055 | INFO    | Flow run 'tiny-marmoset' - Finished in state Completed()

20:23:20.056 | INFO    | Flow run 'portable-malkoha' - Total combinado: 19.5

20:23:20.067 | INFO    | Flow run 'portable-malkoha' - Finished in state Completed()

19.5

En la UI de Prefect, `pipeline-principal` muestra sus dos sub-flows (`procesar-lote`) como runs anidados, cada uno con su propio grafo de tasks. Esto permite ver el tiempo y el estado de cada parte del pipeline por separado.

---
## 7. Estados — qué pasa cuando una task falla

Cada task run tiene un estado: `Running`, `Completed`, `Failed`, `Retrying`, `Crashed`. Por defecto, si una task falla, el flow entero falla. Pero puedes cambiar este comportamiento.

In [8]:
from prefect.futures import PrefectFuture

@task(name="tarea-opcional")
def tarea_que_puede_fallar(valor: int) -> int:
    if valor == 0:
        raise ZeroDivisionError("No se puede dividir por cero")
    return 100 // valor


@flow(name="demo-estados", log_prints=True)
def flow_con_estados():
    # Task que tiene éxito
    resultado_ok = tarea_que_puede_fallar(5)
    print(f"Resultado OK: {resultado_ok}")

    # Task que falla — capturamos la excepción para que el flow continúe
    try:
        resultado_mal = tarea_que_puede_fallar(0)
        print(f"Resultado: {resultado_mal}")
    except ZeroDivisionError as e:
        print(f"Task falló pero el flow continúa: {e}")

    print("Flow completado")


flow_con_estados()

20:23:20.102 | INFO    | Flow run 'bold-clam' - Beginning flow run 'bold-clam' for flow 'demo-estados'

20:23:20.103 | INFO    | Flow run 'bold-clam' - View at http://0.0.0.0:4200/runs/flow-run/d5c9aded-020d-4858-bf6c-581758d8c695

20:23:20.106 | INFO    | Task run 'tarea-opcional-06a' - Finished in state Completed()

20:23:20.108 | INFO    | Flow run 'bold-clam' - Resultado OK: 20

20:23:20.111 | ERROR   | Task run 'tarea-opcional-337' - Task run failed with exception: ZeroDivisionError('No se puede dividir por cero')
Traceback (most recent call last):
  File "/Users/am/Work/personal-page/modulo-mlops/.venv/lib/python3.14/site-packages/prefect/task_engine.py", line 990, in run_context
    yield self
  File "/Users/am/Work/personal-page/modulo-mlops/.venv/lib/python3.14/site-packages/prefect/task_engine.py", line 1644, in run_task_sync
    engine.call_task_fn(txn)
    ~~~~~~~~~~~~~~~~~~~^^^^^
  File "/Users/am/Work/personal-page/modulo-mlops/.venv/lib/python3.14/site-packages/prefect/task_engine.py", line 1007, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
  File "/Users/am/Work/personal-page/modulo-mlops/.venv/lib/python3.14/site-packages/prefect/utilities/callables.py", line 210, in call_with_parameters
    return fn(*args, **kwargs)
  File "/var/folders/lg/tx5z3m_n3md59nq1vd0h_qwr0000gn/T/ipykernel_10451/1455933965.py", line 6, in tarea_que_puede_fallar
    raise ZeroDivisionError("No se puede dividir por cero")
ZeroDivisionError: No se puede dividir por cero

20:23:20.116 | ERROR   | Task run 'tarea-opcional-337' - Finished in state Failed('Task run encountered an exception ZeroDivisionError: No se puede dividir por cero')

20:23:20.117 | INFO    | Flow run 'bold-clam' - Task falló pero el flow continúa: No se puede dividir por cero

20:23:20.118 | INFO    | Flow run 'bold-clam' - Flow completado

20:23:20.129 | INFO    | Flow run 'bold-clam' - Finished in state Completed()

---
## 8. Conectar al servidor Prefect

Todo lo anterior corrió de forma local sin servidor. Prefect lo llama **ephemeral mode**: el flow se ejecuta, pero no hay historial ni UI.

Conectado al servidor, cada flow run queda registrado y es visible en http://localhost:4200.

> Asegúrate de que el stack está levantado antes de ejecutar las celdas siguientes:
> ```bash
> cd infra && docker compose -f docker-compose.yml -f docker-compose.prefect.yml up -d
> ```

In [9]:
import os

# Apuntar al servidor Prefect
os.environ["PREFECT_API_URL"] = "http://localhost:4200/api"

# Verificar conexión
import httpx
resp = httpx.get("http://localhost:4200/api/health")
print(f"Prefect Server status: {resp.json()}")

Prefect Server status: True


In [10]:
@task(name="fetch-datos")
def fetch_datos(n: int) -> list[float]:
    logger = get_run_logger()
    datos = [round(random.gauss(0, 1), 3) for _ in range(n)]
    logger.info(f"Generados {n} datos: media={sum(datos)/len(datos):.3f}")
    return datos


@task(name="calcular-stats")
def calcular_stats(datos: list[float]) -> dict:
    logger = get_run_logger()
    stats = {
        "n":     len(datos),
        "media": round(sum(datos) / len(datos), 4),
        "min":   round(min(datos), 4),
        "max":   round(max(datos), 4),
    }
    logger.info(f"Stats: {stats}")
    return stats


@flow(name="pipeline-estadisticas", log_prints=True)
def pipeline_estadisticas(n_datos: int = 100):
    print(f"Calculando estadísticas para {n_datos} datos")
    datos = fetch_datos(n_datos)
    stats = calcular_stats(datos)
    print(f"Resultado: {stats}")
    return stats


# Este run aparecerá en http://localhost:4200
pipeline_estadisticas(n_datos=50)

20:23:20.188 | INFO    | Flow run 'gorgeous-otter' - Beginning flow run 'gorgeous-otter' for flow 'pipeline-estadisticas'

20:23:20.189 | INFO    | Flow run 'gorgeous-otter' - View at http://0.0.0.0:4200/runs/flow-run/99c992d4-ea57-499a-8439-d4111e863ed3

20:23:20.189 | INFO    | Flow run 'gorgeous-otter' - Calculando estadísticas para 50 datos

20:23:20.193 | INFO    | Task run 'fetch-datos-96f' - Generados 50 datos: media=0.284

20:23:20.195 | INFO    | Task run 'fetch-datos-96f' - Finished in state Completed()

20:23:20.201 | INFO    | Task run 'calcular-stats-89c' - Stats: {'n': 50, 'media': 0.2835, 'min': -2.163, 'max': 3.115}

20:23:20.203 | INFO    | Task run 'calcular-stats-89c' - Finished in state Completed()

20:23:20.204 | INFO    | Flow run 'gorgeous-otter' - Resultado: {'n': 50, 'media': 0.2835, 'min': -2.163, 'max': 3.115}

20:23:20.215 | INFO    | Flow run 'gorgeous-otter' - Finished in state Completed()

{'n': 50, 'media': 0.2835, 'min': -2.163, 'max': 3.115}

**Ir a la UI ahora:** http://localhost:4200 → Flow Runs. Deberías ver `pipeline-estadisticas` con sus dos tasks (`fetch-datos` y `calcular-stats`), los logs de cada una y el tiempo de ejecución.

Ejecuta la celda varias veces con distintos valores de `n_datos` — cada ejecución es un Flow Run independiente en el historial.

In [11]:
# Varios runs con distintos parámetros — observar en la UI
for n in [10, 100, 1000]:
    pipeline_estadisticas(n_datos=n)

20:23:20.245 | INFO    | Flow run 'elated-caiman' - Beginning flow run 'elated-caiman' for flow 'pipeline-estadisticas'

20:23:20.246 | INFO    | Flow run 'elated-caiman' - View at http://0.0.0.0:4200/runs/flow-run/94c1d135-797b-4c11-8bd3-5d54ac764b32

20:23:20.246 | INFO    | Flow run 'elated-caiman' - Calculando estadísticas para 10 datos

20:23:20.250 | INFO    | Task run 'fetch-datos-f69' - Generados 10 datos: media=-0.024

20:23:20.251 | INFO    | Task run 'fetch-datos-f69' - Finished in state Completed()

...

20:23:20.384 | INFO    | Task run 'fetch-datos-cd9' - Finished in state Completed()

20:23:20.393 | INFO    | Task run 'calcular-stats-d91' - Stats: {'n': 1000, 'media': 0.0228, 'min': -3.176, 'max': 3.57}

20:23:20.395 | INFO    | Task run 'calcular-stats-d91' - Finished in state Completed()

20:23:20.398 | INFO    | Flow run 'peridot-kudu' - Resultado: {'n': 1000, 'media': 0.0228, 'min': -3.176, 'max': 3.57}

20:23:20.410 | INFO    | Flow run 'peridot-kudu' - Finished in state Completed()

---
## 9. Inspeccionar runs con PrefectClient

Igual que MLflow tiene `MlflowClient` para acceder a runs por código, Prefect tiene `get_client()` para consultar el servidor de forma programática.

In [12]:
import asyncio
from prefect.client.orchestration import get_client

async def listar_runs(flow_name: str, limit: int = 5):
    async with get_client() as client:
        flows = await client.read_flows()
        flujo = next((f for f in flows if f.name == flow_name), None)
        if not flujo:
            print(f"Flow '{flow_name}' no encontrado")
            return

        runs = await client.read_flow_runs(
            flow_filter=None,
            limit=limit,
            sort="START_TIME_DESC",
        )
        runs_del_flow = [r for r in runs if r.flow_id == flujo.id]

        print(f"Últimos runs de '{flow_name}':")
        for r in runs_del_flow[:limit]:
            duracion = (r.end_time - r.start_time).total_seconds() if r.end_time and r.start_time else "—"
            print(f"  {str(r.id)[:8]}... | {r.state_name:<10} | {duracion}s")

await listar_runs("pipeline-estadisticas")

Últimos runs de 'pipeline-estadisticas':
  0d31d4c0... | Completed  | 0.057116s
  c232f056... | Completed  | 0.033758s
  94c1d135... | Completed  | 0.02536s
  99c992d4... | Completed  | 0.037305s


---
## 10. Resumen

Lo que hemos visto:

| Qué hicimos | Cómo | Para qué |
|-------------|------|----------|
| Definir una task | `@task(name=...)` | Unidad de trabajo con estado y logging |
| Definir un flow | `@flow(name=..., log_prints=True)` | Orquestador de tasks |
| Loggear dentro de tasks | `get_run_logger()` | Logs asociados al task run en la UI |
| Retries automáticos | `@task(retries=3, retry_delay_seconds=5)` | Tolerancia a fallos transitorios |
| Sub-flows | Llamar un `@flow` desde otro `@flow` | Composición y reutilización de pipelines |
| Conectar al servidor | `PREFECT_API_URL=http://localhost:4200/api` | Historial y UI |
| Consultar runs por código | `get_client()` | Inspección programática del servidor |

### Prefect vs MLflow — cuándo usar cada uno

| Pregunta | Herramienta |
|----------|-------------|
| ¿Qué métricas tuvo este entrenamiento? | MLflow |
| ¿Qué modelo está en producción ahora? | MLflow (Model Registry) |
| ¿Cuándo se ejecutó el pipeline por última vez? | Prefect |
| ¿Por qué falló el paso de evaluación el martes? | Prefect |
| ¿A qué hora se lanza el reentrenamiento automático? | Prefect |

Son complementarios: **Prefect gestiona *cuándo* y *cómo* se ejecuta; MLflow registra *qué* pasó**.

### Próximo paso

En el siguiente bloque envolvemos los scripts de `src/` (`train.py`, `evaluate.py`, `register.py`) en tasks de Prefect para que el pipeline completo de churn sea orquestado, observable y reiniciable desde la UI.